In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [3]:
def unstructured_grid_volume(area, depth, surface_elevation, thickness, depth_integrated=False):
    """
    Calculate the volume for every cell in the unstructured grid.

    Parameters
    ----------
    area : np.ndarray
        Element area
    depth : np.ndarray
        Static water depth
    surface_elevation : np.ndarray
        Time-varying surface elevation
    thickness : np.ndarray
        Level (i.e. between layer) position (range 0-1). In FVCOM, this is siglev.
    depth_intergrated : bool, optional
        Set to True to return the depth-integrated volume in addition to the depth-resolved volume. Defaults to False.

    Returns
    -------
    depth_volume : np.ndarray
        Depth-resolved volume of all the elements with time.
    volume : np.ndarray, optional
        Depth-integrated volume of all the elements with time.

    """

    # Convert thickness to actual thickness rather than position in water column of the layer.
    dz = np.abs(np.diff(thickness, axis=0))
    volume = (area * (surface_elevation + depth))
    depth_volume = volume[:, np.newaxis, :] * dz[np.newaxis, ...]

    if depth_integrated:
        return depth_volume, volume
    else:
        return depth_volume

In [4]:
out_nc = xr.open_dataset('../run-output/adamselv_v01_0001_pipeaverage2.nc', decode_times=False)

tracer = out_nc['tracer1_c'].values

x = out_nc['x'].values
y = out_nc['y'].values
tri = np.asarray(out_nc['nv'].values - 1).T
area = out_nc['art1'].values
depth = out_nc['h'].values
surface_elevation = out_nc['zeta'].values
thickness = out_nc['siglev'].values

# time = np.asarray([dt.datetime.strptime(b''.join(rt).decode('utf-8'),'%Y-%m-%dT%H:%M:%S.000000') for rt in out_nc['Times'][:]])

water_col_vol = unstructured_grid_volume(area, depth, surface_elevation, thickness)

tracer_mass_model = water_col_vol*tracer

In [5]:
riv_nc = xr.open_dataset('../04_add_fabm_tracer/riverdata_pipe_tracer_start0.nc', decode_times=False)

In [6]:
riv_nc

<xarray.Dataset> Size: 8MB
Dimensions:      (time: 1193, rivers: 235)
Coordinates:
  * time         (time) float32 5kB 5.632e+04 5.632e+04 ... 5.647e+04 5.647e+04
Dimensions without coordinates: rivers
Data variables:
    Itime        (time) int32 5kB ...
    Itime2       (time) int32 5kB ...
    river_flux   (time, rivers) float32 1MB ...
    river_temp   (time, rivers) float32 1MB ...
    river_salt   (time, rivers) float32 1MB ...
    river_names  (rivers) object 2kB ...
    tracer1_c    (time, rivers) float64 2MB ...
    tracer2_c    (time, rivers) float64 2MB ...
Attributes:
    source:       Akvaplan-niva BuildRiver, version 1.4
    history:      Created 2025-03-27 at 15:55 h by root
    description:  River forcing (temperature and runoff) for FVCOM 4.x

In [7]:
pipe_tracer = riv_nc['tracer1_c'].isel()
pipe_flux = riv_nc['river_flux'][:,-1]

plt.figure(figsize=[8,5])
plt.plot(riv_nc.time, pipe_flux)
plt.ylabel('Pipe flux (m3/s)')
plt.tight_layout()
plt.savefig('pipe_flux.png', dpi=180)
plt.close()

In [8]:
time = out_nc.time.values

In [9]:
time

array([56324. , 56324.5, 56325. , 56325.5])

In [10]:
# Mass conservation correction - fix to the real mass after end of experiment
pipe_c = 10
pipe_discharge = 5.3/60
tracer_mass_discharged = pipe_c*pipe_discharge*(abs(time[0] - time)*86400)

int_tracer_mass = np.sum(np.sum(tracer_mass_model,axis=-1), axis=-1)

In [11]:
tracer_mass_discharged

array([     0.,  38160.,  76320., 114480.])

In [12]:
int_tracer_mass

array([     0.        ,  51349.94446678, 135529.759171  , 222855.986622  ])

In [21]:
yr_vol = pipe_discharge * 86400 * 365  # m3
N_g_m3 = 186305*1000/yr_vol  # 66.9
P_g_m3 = 28622*1000/yr_vol  # 10.3
TOC_g_m3 = 342054*1000/yr_vol  # 122.8

In [20]:
TOC_g_m3

122.79012664771258

In [52]:
calibration = tracer_mass_discharged/int_tracer_mass 
calibration[:1] = 1

tracer_mass = tracer_mass*calibration[:, np.newaxis, np.newaxis]
tracer_conc = tracer_mass/water_col_vol

/var/folders/6l/bpcqsc4x3_ncpql2_wq5f8yw0000gn/T/ipykernel_72275/2239161032.py:1: RuntimeWarning: invalid value encountered in divide
  calibration = tracer_mass_discharged/int_tracer_mass


In [56]:
calibration

array([1.        , 0.74313615, 0.56312356, 0.51369497])

In [ ]:
# Ok rescale to 1 for now
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig,ax = plt.subplots(figsize=[18,14])
ax.plot(time, np.max(np.max(tracer_conc/10, axis=1), axis=1))
ax.set_title('Max tracer concentration anywhere in domain (calibrated to input of 1)')
ax.set_xlabel('Time since start of experiment')
fig.savefig('max_tracer.png')
plt.close()

In [62]:
plot_inds = [1, 2, 3]

pipe_ind = 54469
origin = [x[pipe_ind], y[pipe_ind]]
extent = 1500

for this_t in plot_inds:
    fig,ax = plt.subplots(figsize=[16,16])
    #ax.triplot(x-origin[0],y-origin[1],tri,c='lightgray',linewidth=0.5)
    cm = ax.tripcolor(x-origin[0], y-origin[1], tri,
                      np.max(tracer_conc[this_t,:,:], axis=0), cmap='Purples', edgecolor='k')
    ax.scatter(0, 0, c='r', zorder=2)
    ax.set_xlim([-extent, extent])
    ax.set_ylim([-extent, extent])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    fig.colorbar(cm, cax=cax, orientation='vertical');
    ax.set_xlabel('Distance from pipe (m)')
    ax.set_aspect('equal')
    ax.set_title(f'Max concentration within water column')
    fig.savefig(f'tracer_t{this_t}.png')
    plt.close()